# Feature Engineering

## Objective

The objective of this notebook is to transform the cleaned music dataset into a numerical representation suitable for recommendation modeling.

The feature engineering process focuses on:

- Separating features according to their semantic roles
- Selecting features relevant to content-based recommendation
- Applying appropriate transformations to numerical features
- Constructing a consistent feature matrix for similarity-based recommendation
- Preserving track identifiers and metadata separately from the modeling representation

The resulting feature matrix will be used in the subsequent content-based recommendation system.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
DATA_PATH = "../data/processed/music_clean.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

df.shape

In [ ]:
df.head()

In [ ]:
audio_features = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "valence",
    "tempo"
]

musical_features = [
    "key",
    "mode"
]

track_features = [
    "duration_ms"
]

context_features = [
    "popularity",
    "explicit"
]

## Feature Groups

The available numerical attributes are separated according to their semantic role rather than treating every numerical column as an equivalent audio feature.

### Core audio features

These describe continuous characteristics of the musical content:

- acousticness
- danceability
- energy
- instrumentalness
- liveness
- loudness
- speechiness
- valence
- tempo

### Musical features

`key` and `mode` describe musical properties but have discrete representations.

### Track-level feature

`duration_ms` represents the duration of a track. Although numerical and continuous, its semantic meaning differs from the core audio characteristics.

### Context features

`popularity` and `explicit` provide additional information about a track but are not direct measurements of its acoustic similarity.

These groups will initially be treated separately so that their contribution to recommendation similarity can be evaluated deliberately rather than implicitly.

In [ ]:
content_features = audio_features.copy()

In [ ]:
content_features 

In [ ]:
df[content_features].describe().T

In [ ]:
df[content_features].isna().sum()

In [ ]:
df[content_features].skew().sort_values(ascending=False)

## Feature Distribution Analysis

The skewness of the selected audio features is examined before scaling.

Several features show substantial positive skewness, particularly `speechiness`, `liveness`, and `instrumentalness`. Standardization does not remove this skewness; it only places the features on a comparable scale.

At this stage, no transformation is applied automatically. The effect of potential transformations will be evaluated during model development because transforming a feature can change the geometry of the feature space and therefore affect similarity-based recommendations.

The initial modeling representation will therefore use standardized features as a baseline.

In [ ]:
X = df[content_features].copy()

In [ ]:
X.shape

In [ ]:
X.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_scaled =  scaler.fit_transform(X)

In [ ]:
X_scaled = pd.DataFrame(X_scaled, columns=content_features, index=df.index)

In [ ]:
X_scaled.head()

In [ ]:
X_scaled.describe().T

## Scale Continuous Audio Features

The selected audio features have different numerical ranges. For example, `tempo` is measured in beats per minute, while several other features are normalized to approximately the range 0–1.

Standardization is therefore applied so that features with larger numerical scales do not dominate distance or similarity calculations solely because of their units.

StandardScaler transforms each feature approximately according to:

z = (x - μ) / σ

where μ is the feature mean and σ is the feature standard deviation.

The standardized representation will be used as the initial feature space for similarity-based recommendation.

In [ ]:
X_scaled.shape

In [ ]:
X_scaled.isna().sum().sum()

In [ ]:
np.isfinite(X_scaled).all().all()

In [ ]:
X_scaled.columns.tolist()

In [ ]:
X_scaled.to_csv("../data/processed/audio_features_scaled.csv", index=False)

In [ ]:
track_metadata = df[
    [
        "id",
        "name",
        "artists",
        "year",
        "release_date",
        "popularity",
        "explicit"
    ]
].copy()

In [ ]:
track_metadata.shape

In [ ]:
track_metadata.head()

In [ ]:
track_metadata["id"].nunique()

In [ ]:
track_metadata.to_csv("../data/processed/track_metadata.csv", index=False)

In [ ]:
assert len(X_scaled) == len(track_metadata)
assert X_scaled.index.equals(track_metadata.index)

In [ ]:
assert X_scaled.shape[1] == len(content_features)
assert list(X_scaled.columns) == content_features
assert X_scaled.isna().sum().sum() == 0
assert np.isfinite(X_scaled).all().all()